# Process AR6 Temperature-Limit SLR Inputs for Inequality Analysis

This notebook processes AR6 FACTS sea level rise projections for temperature-limit scenarios
(tlim1.5, tlim2.0, tlim3.0, tlim4.0, tlim5.0) into a format suitable for pyCIAM.

**Inputs:**
- Local SLR: `gs://ar6-lsl-simulations-public-standard/gridded/full_sample_workflows/{workflow}/{tlim}win0.25/total-workflow.zarr`
- Global SLR: `gs://impactlab-data/coastal/data/raw/slr/ar6/ar6/global/full_sample_workflows/{workflow}/{tlim}win0.25/total-workflow.nc`
- VLM: `gs://ar6-lsl-simulations-requesterpays-standard/gridded/full_sample_components/verticallandmotion-kopp14-verticallandmotion_localsl.zarr`

**Output:**
- Processed SLR zarr with dimensions: `(scenario[5], year[9], sample[1000], site_id[~50000])`
- Variables: `lsl_msl05`, `lsl_ncc_msl05`, `gsl_msl05`, `lat`, `lon`

In [ ]:
import sys
sys.path.append("../..")

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from itertools import product
from gcsfs import GCSFileSystem

from config import (
    DIR_SLR_AR6_RAW,
    DIR_SLR_AR6_GRIDDED_PUBLIC,
    PATH_VLM_REQUESTER_PAYS,
    PATH_SLR_INEQUALITY,
    TLIM_SCENARIOS,
    WORKFLOWS,
    N_SAMPLES_PER_WORKFLOW,
    N_SAMPLES_TOTAL,
    save_zarr,
)

## Setup Dask Cluster

In [ ]:
import os
from dask_gateway import Gateway

img = os.environ.get("JUPYTER_IMAGE", None)

gateway = Gateway()
cluster = gateway.new_cluster(
    idle_timeout=900,
    profile="micro",
    **(dict(worker_image=img, scheduler_image=img) if img else {})
)

client = cluster.get_client()
cluster.adapt(minimum=10, maximum=500)
cluster

## Helper Functions

In [ ]:
def open_and_convert_zarr(ds_path):
    """Open zarr and convert SLR units to meters."""
    out = xr.open_zarr(ds_path)
    out["sea_level_change"] = (
        out.sea_level_change.pint.quantify().pint.to("meters").pint.dequantify()
    )
    return out


def open_and_convert_nc(ds_path):
    """Open netCDF and convert SLR units to meters."""
    # Convert gs:// to /gcs/ fuse path
    _path = str(ds_path).replace("gs://", "/gcs/")
    out = xr.open_dataset(_path)
    out["sea_level_change"] = (
        out.sea_level_change.pint.quantify().pint.to("meters").pint.dequantify()
    )
    return out

## Process Local and Global SLR for Temperature-Limit Scenarios

For each tlim scenario and workflow:
1. Load 20,000 MC samples
2. Downsample to 500 quantiles per workflow (1000 total)
3. Stack workflows together

In [ ]:
# Set random seed for reproducibility (same as original AR6.ipynb)
np.random.seed(11222023)

# Generate quantile indices (500 per workflow)
nsamps = 20000
low = np.arange(0, nsamps, step=nsamps / N_SAMPLES_PER_WORKFLOW)
high = np.arange(0, nsamps, step=nsamps / N_SAMPLES_PER_WORKFLOW) + nsamps / N_SAMPLES_PER_WORKFLOW
quants = np.random.randint(low=low, high=high, size=None) / nsamps

print(f"Number of quantiles per workflow: {len(quants)}")
print(f"Quantile range: {quants.min():.4f} to {quants.max():.4f}")

In [ ]:
# Process local (gridded) SLR and global SLR for each tlim/workflow combination
local_dfs = []
global_dfs = []

for tlim, workflow in product(TLIM_SCENARIOS, WORKFLOWS):
    print(f"Processing {tlim} / {workflow}...")
    
    # Local (gridded) SLR from public bucket
    local_path = f"{DIR_SLR_AR6_GRIDDED_PUBLIC}/{workflow}/{tlim}win0.25/total-workflow.zarr"
    df = open_and_convert_zarr(local_path)
    df = (
        df.sel(years=slice(2020, 2100), drop=True)
        .sea_level_change.chunk(dict(samples=-1))
        .quantile(quants, dim='samples')
        .assign_coords({'workflow': workflow, 'tlim': tlim})
        .expand_dims(['workflow', 'tlim'])
    )
    local_dfs.append(df)
    
    # Global SLR from impactlab bucket
    global_path = DIR_SLR_AR6_RAW / workflow / f"{tlim}win0.25" / "total-workflow.nc"
    global_df = open_and_convert_nc(global_path)
    global_df = (
        global_df.sel(years=slice(2020, 2100), drop=True)
        .sea_level_change.chunk(dict(samples=-1))
        .quantile(quants, dim='samples')
        .assign_coords({'workflow': workflow, 'tlim': tlim})
        .expand_dims(['workflow', 'tlim'])
    )
    global_dfs.append(global_df)

print("Done processing all scenarios.")

In [ ]:
# Combine local SLR across scenarios and workflows
# Stack workflows into sample dimension (500 × 2 = 1000 samples)
df_full = (
    xr.combine_by_coords(local_dfs)
    .stack(samples=['workflow', 'quantile'])
    .assign_coords({'samples': np.arange(1, N_SAMPLES_TOTAL + 1)})
)

# Combine global SLR
global_ds = (
    xr.combine_by_coords(global_dfs)
    .stack(samples=['workflow', 'quantile'])
    .assign_coords({'samples': np.arange(1, N_SAMPLES_TOTAL + 1)})
    .squeeze(drop=True)
    .sea_level_change
)

print(f"Local SLR shape: {df_full.dims}")
print(f"Global SLR shape: {global_ds.dims}")

## Process VLM (No-Climate-Change Component)

The VLM component represents vertical land motion, which is the "no climate change" counterfactual.

**Note:** This requires access to the requester-pays bucket. If running locally, this will fail.
Run this notebook on the compute cluster where requester-pays access is configured.

In [ ]:
# Access VLM from requester-pays bucket
fs = GCSFileSystem(requester_pays=True)
mapping = fs.get_mapper(PATH_VLM_REQUESTER_PAYS)

# Generate quantiles for VLM (same seed, but 1000 samples directly)
np.random.seed(11222023)
nsamps = 20000
low = np.arange(0, nsamps, step=nsamps / N_SAMPLES_TOTAL)
high = np.arange(0, nsamps, step=nsamps / N_SAMPLES_TOTAL) + nsamps / N_SAMPLES_TOTAL
quants_vlm = np.random.randint(low=low, high=high, size=None) / nsamps

# Load and process VLM
vlm_df = open_and_convert_zarr(mapping)
vlm_df = (
    vlm_df.sel(years=slice(2020, 2100), drop=True)
    .sea_level_change.chunk(dict(samples=-1))
    .quantile(quants_vlm, dim='samples')
    .rename({'quantile': 'samples'})
    .assign_coords({'samples': np.arange(1, N_SAMPLES_TOTAL + 1)})
    .to_dataset()
)

print(f"VLM shape: {vlm_df.dims}")

## Combine into Final Dataset

In [ ]:
# Handle floating point matching errors on samples dimension
global_ds["samples"] = df_full.samples
vlm_df["samples"] = df_full.samples

# Create combined dataset
all_ds = xr.Dataset(
    {
        "lsl_msl05": df_full.sea_level_change.persist(),
        "lsl_ncc_msl05": vlm_df.sea_level_change.persist(),
        "gsl_msl05": global_ds.persist(),
        "lon": vlm_df.lon,
        "lat": df_full.lat,
    }
)

# Handle -180 longitude
all_ds["lon"] = all_ds.lon.where(all_ds.lon != -180, 180)

# Stack lat/lon into locations
all_ds = all_ds.stack(locations=['lat', 'lon']).persist()

print(all_ds)

In [ ]:
# Drop locations with NaN values
valid = (
    all_ds[["lsl_msl05", "lsl_ncc_msl05"]]
    .sel(years=slice(2100))
    .notnull()
    .all(["tlim", "samples", "years"])
    .to_array("tmp")
    .all("tmp")
).persist()

all_ds = all_ds.sel(locations=valid.where(valid, drop=True).locations)

# Rename dimensions to pyCIAM conventions
all_ds = all_ds.rename(
    {"years": "year", "samples": "sample", "locations": "site_id", "tlim": "scenario"}
)

# Verify no missing values
assert all_ds.sel(year=slice(2100)).notnull().all().to_array().all()

print(f"Final dataset shape: {all_ds.dims}")
print(all_ds)

In [ ]:
# Set optimal chunking for pyCIAM
all_ds = all_ds.chunk({'site_id': -1, 'scenario': 1, 'year': -1, 'sample': 100})
all_ds

## Save Processed SLR Data

In [ ]:
# Save to zarr
print(f"Saving to {PATH_SLR_INEQUALITY}...")
save_zarr(all_ds, PATH_SLR_INEQUALITY, mode='w')
print("Done!")

In [ ]:
# Verify saved data
verify = xr.open_zarr(str(PATH_SLR_INEQUALITY))
print(verify)
print(f"\nScenarios: {verify.scenario.values}")
print(f"Years: {verify.year.values}")
print(f"Samples: {verify.sample.values[:5]}...{verify.sample.values[-5:]}")
print(f"Number of sites: {len(verify.site_id)}")

In [ ]:
# Cleanup
client.close()
cluster.close()